# 📓 CUADERNO 02: Integración de Firmas de Expresión CMap (L1000) y Modulación Dinámica del Interactoma
### Plan de Beca FaCENA 2025-2026 - Módulo 2 (Actividad A2 / OE2)
**Becaria:** Anahí Ayelén Espinoza | **Directores:** Dr. Emilio Angelina & Dirección de Beca FaCENA UNNE

---

### 🎯 Objetivos de este Cuaderno:
1. **Cargar el Grafo PPI** generado en el Cuaderno 01 ($G = (V, E, X)$).
2. **Ingestar Firmas de Expresión Differential (FEs)** de **Connectivity Map (CMap L1000)** compuestas por vectores de z-scores de expresión génica.
3. **Definir la Firma de Referencia Terapéutica** (ej. Knockout CRISPR de un gen driver en cáncer como *BRAF*).
4. **Ingestar Firmas de Productos Naturales** (COCONUT / CMap: Curcumina, Resveratrol, Quercetina, EGCG, etc.).
5. **Implementar el Algoritmo de Modulación Dinámica del Grafo**:
   $$A_{ij}^{(pert)} = A_{ij} \cdot \exp(-\alpha |z_i - z_j|)$$
6. **Exportar el Dataset de Perturbaciones** (`graph_perturbations_dataset.pt`).


In [3]:
# ==============================================================================
# PASO 0: Carga de Librerías y Dataset Previos
# ==============================================================================
import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Install torch_geometric if not already installed
try:
    import torch_geometric
except ImportError:
    !pip install torch_geometric
    import torch_geometric


base_dir = "./data"
os.makedirs(base_dir, exist_ok=True)
data_path = os.path.join(base_dir, "ppi_human_esm2.pt")
if not os.path.exists(data_path) and os.path.exists("ppi_human_esm2.pt"):
    data_path = "ppi_human_esm2.pt"

if os.path.exists(data_path):
    saved_data = torch.load(data_path)
    grafo_base = saved_data["grafo_pyg"]
    gene_to_idx = saved_data["gene_to_idx"]
    idx_to_gene = saved_data["idx_to_gene"]
    print("✅ Grafo PPI base y embeddings ESM-2 cargados exitosamente.")
else:
    print("⚠️ Creando datos base in-memory...")
    genes = ["BRAF", "RAF1", "MAP2K1", "MAPK1", "TP53", "KRAS", "EGFR", "PIK3CA", "AKT1", "MTOR"]
    gene_to_idx = {g: i for i, g in enumerate(genes)}
    idx_to_gene = {i: g for i, g in enumerate(genes)}
    X = torch.randn(len(genes), 640)
    edge_index = torch.tensor([[0, 1, 1, 2, 3, 4], [1, 0, 2, 1, 4, 3]], dtype=torch.long)
    edge_attr = torch.tensor([[0.9], [0.9], [0.8], [0.8], [0.7], [0.7]], dtype=torch.float)
    from torch_geometric.data import Data
    grafo_base = Data(x=X, edge_index=edge_index, edge_attr=edge_attr)
    grafo_base.gene_names = genes

print(f" • Nodos (|V|): {grafo_base.x.shape[0]}")
print(f" • Aristas (|E|): {grafo_base.edge_index.shape[1] // 2}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 16.8 MB/s eta 0:00:00
⚠️ Creando datos base in-memory...
 • Nodos (|V|): 10
 • Aristas (|E|): 3


## 🧪 PASO 1: Simulación e Ingesta de Firmas de Expresión (CMap L1000 z-scores)


In [4]:
np.random.seed(42)
nodos_genes = [idx_to_gene[i] for i in range(len(idx_to_gene))]

# 1. FIRMA DE REFERENCIA TERAPÉUTICA: CRISPR KO del gen driver BRAF
z_ref_braf_ko = {gene: 0.0 for gene in nodos_genes}
z_ref_braf_ko["BRAF"] = -3.8
z_ref_braf_ko["MAP2K1"] = -2.5
z_ref_braf_ko["MAPK1"] = -2.1
z_ref_braf_ko["TP53"] = +1.5

# 2. QUIOMIOTECA DE PRODUCTOS NATURALES (COCONUT / CMap)
productos_naturales = {
    "Curcumina": {"BRAF": -2.8, "MAP2K1": -1.9, "TP53": +1.2, "AKT1": -1.5, "TNF": -2.1},
    "Resveratrol": {"BRAF": -2.1, "MAPK1": -2.0, "TP53": +1.8, "MTOR": -2.4, "PTEN": +1.6},
    "Quercetina": {"BRAF": -1.5, "EGFR": -2.2, "PIK3CA": -1.8, "AKT1": -2.0},
    "EGCG (Té Verde)": {"BRAF": -2.9, "MAP2K1": -2.3, "MAPK1": -1.8, "TP53": +1.4},
    "Berberina": {"KRAS": -2.0, "AKT1": -2.2, "MTOR": -1.9, "TP53": +1.1},
    "Decoy_Inactivo_1": {"BRAF": +0.2, "MAP2K1": -0.1, "TP53": -0.3, "AKT1": +0.1},
    "Decoy_Inactivo_2": {"EGFR": +0.5, "KRAS": +0.4, "PIK3CA": +0.2}
}

def obtener_vector_z(dict_z, lista_genes):
    return torch.tensor([dict_z.get(g, np.random.normal(0, 0.1)) for g in lista_genes], dtype=torch.float)

vector_z_ref = obtener_vector_z(z_ref_braf_ko, nodos_genes)
print("✅ Vector z-score de Referencia Terapéutica (BRAF KO):")
for g, z in zip(nodos_genes[:5], vector_z_ref[:5]):
    print(f"   - {g:8s}: z = {z.item():+.2f}")

✅ Vector z-score de Referencia Terapéutica (BRAF KO):
   - BRAF    : z = -3.80
   - RAF1    : z = +0.00
   - MAP2K1  : z = -2.50
   - MAPK1   : z = -2.10
   - TP53    : z = +1.50


## ⚡ PASO 2: Algoritmo de Modulación Dinámica del Grafo ($A_{pert}$)


In [5]:
def modular_interactoma_dinamico(grafo_base, vector_z, alpha=0.3):
    edge_index = grafo_base.edge_index
    edge_attr = grafo_base.edge_attr.clone()

    src = edge_index[0]
    tgt = edge_index[1]

    z_src = vector_z[src]
    z_tgt = vector_z[tgt]

    delta_z = torch.abs(z_src - z_tgt)
    factor_modulacion = torch.exp(-alpha * delta_z).unsqueeze(1)

    new_edge_attr = edge_attr * factor_modulacion

    shift_nodal = vector_z.unsqueeze(1) * 0.05
    new_x = grafo_base.x + shift_nodal

    from torch_geometric.data import Data
    grafo_pert = Data(x=new_x, edge_index=edge_index, edge_attr=new_edge_attr)
    grafo_pert.gene_names = getattr(grafo_base, "gene_names", None)
    return grafo_pert

grafo_ref_braf = modular_interactoma_dinamico(grafo_base, vector_z_ref)
print("✅ Grafo modulado de Referencia Terapéutica (BRAF KO) construido:")
print(f" • Rango de pesos originales : [{grafo_base.edge_attr.min():.3f}, {grafo_base.edge_attr.max():.3f}]")
print(f" • Rango de pesos modulados  : [{grafo_ref_braf.edge_attr.min():.3f}, {grafo_ref_braf.edge_attr.max():.3f}]")

✅ Grafo modulado de Referencia Terapéutica (BRAF KO) construido:
 • Rango de pesos originales : [0.700, 0.900]
 • Rango de pesos modulados  : [0.238, 0.378]


## 🌿 PASO 3: Modulación para Toda la Quimioteca de Productos Naturales


In [7]:
dict_grafos_productos = {}

for nombre_compuesto, dict_z in productos_naturales.items():
    v_z = obtener_vector_z(dict_z, nodos_genes)
    g_pert = modular_interactoma_dinamico(grafo_base, v_z)
    dict_grafos_productos[nombre_compuesto] = g_pert

print(f"✅ Se construyeron {len(dict_grafos_productos)} grafos modulados para la quimioteca.")

output_dataset = {
    "grafo_wt": grafo_base,
    "grafo_ref_braf": grafo_ref_braf,
    "dict_grafos_productos": dict_grafos_productos,
    "nodos_genes": nodos_genes,
    "gene_to_idx": gene_to_idx,
    "idx_to_gene": idx_to_gene
}

base_dir = "./data"
os.makedirs(base_dir, exist_ok=True)
save_path = os.path.join(base_dir, "graph_perturbations_dataset.pt")
torch.save(output_dataset, save_path)
print(f"💾 Archivo '{save_path}' guardado exitosamente para el Cuaderno 03.")


✅ Se construyeron 7 grafos modulados para la quimioteca.
💾 Archivo './data/graph_perturbations_dataset.pt' guardado exitosamente para el Cuaderno 03.
